# NLP Lab — Text Classification — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



9 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np
import re
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup
*Section: Setup*

Setup

In [ ]:
import math
import numpy as np
import re
from collections import Counter

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 2: Section 0 — Dataset split
*Section: Section 0*

## Section 0 — Dataset (offline)

This lab uses a small offline corpus with 3 labels: `sports`, `politics`, and `tech`.

FAANG gotcha: make the split **deterministic and stratified**, and fit every text featurizer on **train only**.

### Task 0.1 — Split
Create a deterministic 75/25 stratified train/val split using `rng.permutation` within each class.

In [ ]:
$$10

In [ ]:
# Checks
check('disjoint', set(train_idx).isdisjoint(set(val_idx)))
check('balanced_split', len(np.unique(y_train)) == len(label_names) and len(np.unique(y_val)) == len(label_names))

## Task 3: Section 0 — Metrics
*Section: Section 0*

### Task 0.2 — Metrics
Implement classification accuracy and a `k x k` confusion matrix.

In [ ]:
def accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float((y_true == y_pred).mean())

def confusion_k(y_true: np.ndarray, y_pred: np.ndarray, k: int) -> np.ndarray:
    cm = np.zeros((k, k), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

## Task 4: Sections 1-2 — Tokenization + TF-IDF features
*Section: Sections 1-2*

## Section 1 — Tokenization

### Task 1.1
Implement a simple tokenizer that returns lowercase alphabetic tokens.

## Section 2 — TF-IDF features

### Task 2.1
Implement `build_vocab`, count vectorization, and TF-IDF fitting using **training documents only**.

In [ ]:
$$11

In [ ]:
# Checks
check('vocab_nonempty', len(vocab) > 0)
check('tfidf_shape', Xtr.shape[0] == len(X_train_text) and Xtr.shape[1] == len(vocab))

## Task 5: Section 3 — Softmax Logistic Regression
*Section: Section 3*

## Section 3 — Softmax Logistic Regression (multiclass)

### Task 3.1
Implement softmax + cross-entropy and train a multiclass linear classifier on the TF-IDF features.

In [ ]:
def softmax(Z: np.ndarray) -> np.ndarray:
    Z = Z - np.max(Z, axis=1, keepdims=True)
    E = np.exp(Z)
    return E / np.sum(E, axis=1, keepdims=True)

def fit_softmax_lr(X: np.ndarray, y: np.ndarray, k: int, lr: float = 0.6, steps: int = 1200, l2: float = 1e-3, seed: int = 0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    W = 0.01 * rng.standard_normal((d, k))
    b = np.zeros((k,), dtype=np.float64)
    Y = np.eye(k)[y]
    for _ in range(steps):
        P = softmax(X @ W + b)
        gradW = (X.T @ (P - Y)) / n + l2 * W
        gradb = (P - Y).mean(axis=0)
        W -= lr * gradW
        b -= lr * gradb
    return W, b

def predict_softmax_lr(X: np.ndarray, W: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.argmax(X @ W + b, axis=1).astype(np.int64)

W, b = fit_softmax_lr(Xtr, y_train, k=3)
yhat = predict_softmax_lr(Xva, W, b)
print('TFIDF+LR val_acc', accuracy(y_val, yhat))
print(confusion_k(y_val, yhat, 3))

In [ ]:
# Checks
check('lr_pred_shape', yhat.shape == y_val.shape)

## Task 6: Section 4.1 — PPMI + SVD word embeddings
*Section: Section 4*

## Section 4 — Word embeddings from train only

Here you will learn a small unsupervised word-vector pipeline from the training corpus only: co-occurrence matrix -> PPMI -> SVD.

### Task 4.1
Implement co-occurrence, PPMI, and SVD embeddings.

In [ ]:
# No source solution — left as an exercise.

## Task 7: Section 4.2 — Document vectors
*Section: Section 4*

### Task 4.2 — Document vectors
Turn each document into a dense vector by averaging its word embeddings, ignoring `UNK` when possible.

In [ ]:
def doc_embed(texts: list[str], w2i: dict[str, int], Ew: np.ndarray) -> np.ndarray:
    d = Ew.shape[1]
    X = np.zeros((len(texts), d), dtype=np.float64)
    for i, s in enumerate(texts):
        ids = [wid(w, w2i) for w in tokenize(s)]
        ids = [j for j in ids if j != 0]
        if ids:
            X[i] = Ew[ids].mean(axis=0)
    X = X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)
    return X

Xtr_e = doc_embed(X_train_text, w2i, Ew)
Xva_e = doc_embed(X_val_text, w2i, Ew)
check('doc_shape', Xtr_e.shape[0] == len(X_train_text) and Xva_e.shape[0] == len(X_val_text))
check('doc_dim', Xtr_e.shape[1] == Ew.shape[1])
check('doc_norm', np.all(np.linalg.norm(Xtr_e, axis=1) > 0))

In [ ]:
# Checks
check('doc_norm', float(np.max(np.abs(np.linalg.norm(Xtr_e, axis=1) - 1.0))) < 1e-6)

## Task 8: Section 5 — Dense document classifier
*Section: Section 5*

## Section 5 — Train classifier on dense document vectors

### Task 5.1
Train the same softmax logistic-regression model on the averaged document embeddings and compare it with the TF-IDF baseline.

In [ ]:
W2, b2 = fit_softmax_lr(Xtr_e, y_train, k=3, lr=0.6, steps=1200, l2=1e-3, seed=0)
yhat2 = predict_softmax_lr(Xva_e, W2, b2)
print('EmbedAvg+LR val_acc', accuracy(y_val, yhat2))
print(confusion_k(y_val, yhat2, 3))

**Why this works.** Averaging dense word vectors compresses sparse lexical features into a smaller semantic representation. On tiny datasets, this can help with similarity generalization, but it can also lose exact keyword signals that TF-IDF preserves.

## Task 9: Section 6 — Error analysis
*Section: Section 6*

## Section 6 — Error analysis

### Task 6.1
Print misclassified examples for both models and inspect what kinds of mistakes the sparse and dense pipelines make.

In [ ]:
def show_mistakes(texts: list[str], y_true: np.ndarray, y_pred: np.ndarray, label_names: list[str], max_n: int = 10):
    n = 0
    for t, yt, yp in zip(texts, y_true, y_pred):
        if int(yt) != int(yp):
            print('true=', label_names[int(yt)], 'pred=', label_names[int(yp)], '::', t)
            n += 1
            if n >= max_n:
                break

print('--- TFIDF mistakes ---')
show_mistakes(X_val_text, y_val, yhat, label_names)
print('--- EmbedAvg mistakes ---')
show_mistakes(X_val_text, y_val, yhat2, label_names)